<a href="https://colab.research.google.com/github/1rishu0/Computer-Vision/blob/main/Computer_Vision_Masterclass_Deep_dream.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Masterclass: Deep Dream

## Importing the libraries

- Adapted from: https://www.tensorflow.org/beta/tutorials/generative/deepdream

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
tf.__version__

## Loading the pre-built convolutional neural network

- InceptionNet: https://www.tensorflow.org/api_docs/python/tf/keras/applications/inception_v3
- Original paper: https://www.cv-foundation.org/openaccess/content_cvpr_2016/papers/Szegedy_Rethinking_the_Inception_CVPR_2016_paper.pdf
- Imagenet: http://www.image-net.org/

In [ ]:
base_model = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet')

In [ ]:
base_model.summary()

In [ ]:
len(base_model.layers)

In [ ]:
# Relu
#names = ['mixed3', 'mixed5', 'mixed8', 'mixed9']
names = ['mixed3', 'mixed5']

In [ ]:
base_model.input

In [ ]:
layers = [base_model.get_layer(name).output for name in names]

In [ ]:
layers

In [ ]:
deep_dream_model = tf.keras.Model(inputs = base_model.input, outputs = layers)

## Loading and pre-processing the image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
image = tf.keras.preprocessing.image.load_img('/content/drive/MyDrive/Cursos - recursos/Computer Vision Masterclass/Images/StaryNight.jpg',
                                              target_size=(225, 375))

In [ ]:
plt.imshow(image);

In [ ]:
type(image)

In [ ]:
image.size

In [ ]:
image.mode, len(image.mode)

In [ ]:
list(image.getdata())

In [ ]:
image = tf.keras.preprocessing.image.img_to_array(image)

In [ ]:
type(image)

In [ ]:
image.shape

In [ ]:
image.min(), image.max()

In [ ]:
# image = image / 255
image = tf.keras.applications.inception_v3.preprocess_input(image)

In [ ]:
image.min(), image.max()

## Getting the activations

In [ ]:
image.shape

In [ ]:
image_batch = tf.expand_dims(image, axis = 0)

In [ ]:
image_batch.shape

In [ ]:
activations = deep_dream_model.predict(image_batch)

In [ ]:
deep_dream_model.outputs

In [ ]:
len(activations)

In [ ]:
activations[1]

In [ ]:
activations[0].shape, activations[1].shape

## Calculating the loss

In [ ]:
def calculate_loss(image, network):
  image_batch = tf.expand_dims(image, axis = 0)
  activations = network(image_batch)

  losses = []
  for act in activations:
    loss = tf.math.reduce_mean(act)
    losses.append(loss)

  #print(losses)
  #print(np.shape(losses))
  #print(tf.reduce_sum(losses))

  return tf.reduce_sum(losses)

In [ ]:
0.45195404 + 0.16485049

In [ ]:
loss = calculate_loss(image, deep_dream_model)
loss

## Gradient ascent

In [ ]:
# Compare the activations with the pixels
# Emphasize parts of the image
# Change the pixels of the input image

@tf.function
def deep_dream(network, image, learning_rate):
  with tf.GradientTape() as tape:
    tape.watch(image)
    loss = calculate_loss(image, network)

  gradients = tape.gradient(loss, image) # Derivate
  gradients /= tf.math.reduce_std(gradients)
  image = image + gradients * learning_rate
  image = tf.clip_by_value(image, -1, 1)

  return loss, image

In [ ]:
def inverse_transform(image):
  image = 255 * (image + 1.0) / 2.0
  return tf.cast(image, tf.uint8)

In [ ]:
def run_deep_dream(network, image, epochs, learning_rate):
  for epoch in range(epochs):
    loss, image = deep_dream(network, image, learning_rate)

    if epoch % 200 == 0:
      plt.figure(figsize=(12,12))
      plt.imshow(inverse_transform(image))
      plt.show()
      print('Epoch {}, loss {}'.format(epoch, loss))

## Generating images

In [ ]:
image.shape, type(image)

In [ ]:
run_deep_dream(network=deep_dream_model, image=image, epochs = 8000, learning_rate=0.0001)

## Homework

In [ ]:
image = tf.keras.preprocessing.image.load_img('/content/drive/MyDrive/Cursos - recursos/Computer Vision Masterclass/Images/sky.jpeg',
                                              target_size = (225, 375))

In [ ]:
plt.imshow(image);

In [ ]:
image = tf.keras.preprocessing.image.img_to_array(image)
image = tf.keras.applications.inception_v3.preprocess_input(image)

In [ ]:
run_deep_dream(network = deep_dream_model, image = image, epochs = 8000, learning_rate = 0.0001)